In [3]:
import pulp

problem = pulp.LpProblem("SLE", pulp.LpMaximize)

x = pulp.LpVariable("x", cat="Continuous")
y = pulp.LpVariable("y", cat="Continuous")

problem += 120 * x + 150 * y == 1440
problem += x + y == 10

status = problem.solve()

print("Status:", pulp.LpStatus[status])
print('x=', x.value(), 'y=', y.value())

Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /Users/miyakoh/dev/tech_books/PyOptBook-main/.venv/lib/python3.12/site-packages/pulp/solverdir/cbc/osx/64/cbc /var/folders/fk/j0dk8tns78g4y3w9yv61yf780000gn/T/614c0bcc3b12486292a8a4b3709db52f-pulp.mps -max -timeMode elapsed -branch -printingOptions all -solution /var/folders/fk/j0dk8tns78g4y3w9yv61yf780000gn/T/614c0bcc3b12486292a8a4b3709db52f-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 7 COLUMNS
At line 13 RHS
At line 16 BOUNDS
At line 20 ENDATA
Problem MODEL has 2 rows, 3 columns and 4 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Presolve 0 (-2) rows, 0 (-3) columns and 0 (-4) elements
Empty problem - 0 rows, 0 columns and 0 elements
Optimal - objective value -0
After Postsolve, objective 0, infeasibilities - dual 0 (0), primal 0 (0)
Optimal objective 0 - 0 iterations time 0.002, Presolve 0.00
Option for printin

In [ ]:
import pandas as pd 

require_df = pd.read_csv('requires.csv')
require_df

,p,m,require
0,p1,m1,2
1,p1,m2,0
2,p1,m3,1
3,p2,m1,3
4,p2,m2,2
5,p2,m3,0
6,p3,m1,0
7,p3,m2,2
8,p3,m3,2
9,p4,m1,2


In [26]:
gain_df = pd.read_csv('gains.csv')
gain_df

,p,gain
0,p1,3
1,p2,4
2,p3,4
3,p4,5


In [6]:
stock_df = pd.read_csv('stocks.csv')
stock_df

,m,stock
0,m1,35
1,m2,22
2,m3,27


In [8]:
P = gain_df['p'].tolist()
print(P)

['p1', 'p2', 'p3', 'p4']


In [10]:
M = stock_df['m'].to_list()
print(M)

['m1', 'm2', 'm3']


In [14]:
stock = {row.m:row.stock for row in stock_df.itertuples()}
print(stock)

{'m1': 35, 'm2': 22, 'm3': 27}


In [24]:
require = {(row.p, row.m):row.require for row in require_df.itertuples()}
print(require)

{('p1', 'm1'): 2, ('p1', 'm2'): 0, ('p1', 'm3'): 1, ('p2', 'm1'): 3, ('p2', 'm2'): 2, ('p2', 'm3'): 0, ('p3', 'm1'): 0, ('p3', 'm2'): 2, ('p3', 'm3'): 2, ('p4', 'm1'): 2, ('p4', 'm2'): 2, ('p4', 'm3'): 2}


In [25]:
gain = {row.p:row.gain for row in gain_df.itertuples()}
print(gain)

{'p1': 3, 'p2': 4, 'p3': 4, 'p4': 5}


In [27]:
problem = pulp.LpProblem('LP2', pulp.LpMaximize)

In [28]:
x = pulp.LpVariable.dicts('x', P, cat='Continuous')
for p in P:
    problem += x[p] >= 0

for m in M:
    problem += pulp.lpSum([require[p, m] * x[p] for p in P]) <= stock[m]

In [29]:
problem += pulp.lpSum([gain[p] * x[p] for p in P])

In [30]:
status = problem.solve()
print('Status:', pulp.LpStatus[status])

Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /Users/miyakoh/dev/tech_books/PyOptBook-main/.venv/lib/python3.12/site-packages/pulp/solverdir/cbc/osx/64/cbc /var/folders/fk/j0dk8tns78g4y3w9yv61yf780000gn/T/9cca064a8e8f4d80a340cec7f54da113-pulp.mps -max -timeMode elapsed -branch -printingOptions all -solution /var/folders/fk/j0dk8tns78g4y3w9yv61yf780000gn/T/9cca064a8e8f4d80a340cec7f54da113-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 12 COLUMNS
At line 30 RHS
At line 38 BOUNDS
At line 43 ENDATA
Problem MODEL has 7 rows, 4 columns and 13 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Presolve 3 (-4) rows, 4 (0) columns and 9 (-4) elements
0  Obj -0 Dual inf 17.499996 (4)
4  Obj 80.428571
Optimal - objective value 80.428571
After Postsolve, objective 80.428571, infeasibilities - dual 0 (0), primal 0 (0)
Optimal objective 80.42857143 - 4 iterations time 0.002, Preso

In [ ]:
for p in P:
    print(p, x[p].value())

print('obj=', problem.objective.value)

p1 12.142857
p2 3.5714286
p3 7.4285714
p4 0.0
obj= 80.42857099999999


NoneType

In [42]:
type(x['p1'])

pulp.pulp.LpVariable

In [44]:
x = pulp.LpVariable.dicts('x', P, cat='Integer')
problem = pulp.LpProblem('IP', pulp.LpMaximize)

for p in P:
    problem += x[p] >= 0

for m in M:
    problem += pulp.lpSum([require[p, m] * x[p] for p in P]) <= stock[m]

problem += pulp.lpSum([gain[p] * x[p] for p in P])
status = problem.solve()
print('Status:', pulp.LpStatus[status])

for p in P:
    print(p, x[p].value())

print('obj=', problem.objective.value())

Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /Users/miyakoh/dev/tech_books/PyOptBook-main/.venv/lib/python3.12/site-packages/pulp/solverdir/cbc/osx/64/cbc /var/folders/fk/j0dk8tns78g4y3w9yv61yf780000gn/T/50a57151643e491aacff8974a4c0d94e-pulp.mps -max -timeMode elapsed -branch -printingOptions all -solution /var/folders/fk/j0dk8tns78g4y3w9yv61yf780000gn/T/50a57151643e491aacff8974a4c0d94e-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 12 COLUMNS
At line 38 RHS
At line 46 BOUNDS
At line 51 ENDATA
Problem MODEL has 7 rows, 4 columns and 13 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Continuous objective value is 80.4286 - 0.00 seconds
Cgl0004I processed model has 3 rows, 4 columns (4 integer (0 of which binary)) and 9 elements
Cutoff increment increased from 1e-05 to 0.9999
Cbc0012I Integer solution of -76 found by DiveCoefficient after 0 iterations and 0 nodes (